# Data Merge Tutorial 1: Merging KCMO Building Datasets
### Case Study: Matching KC Building Records Across Files

## Introduction

Real-world administrative datasets are rarely ready to merge perfectly. Even when two files describe the same buildings, identifiers may be missing, addresses may be formatted differently, and field names may not line up exactly.

In this tutorial series, we will build a step-by-step workflow for exploring, cleaning, and merging datasets across years. The tutorial was built around Kansas City Missouri building datasets from 2025 and 2026.

To merge datasets we often need to:

1. explore each dataset,
2. check missingness and data quality,
3. standardize key fields,
4. merge exact matches first,
5. and then use fuzzy matching for records that still remain unmatched.

## Tutorial Plan

We will:

1. Load/simulate? 2025 and 2026 building records.
2. Standardize a small set of columns into a shared schema.
3. Create exact-match keys.
4. Join rows across years using those exact keys.
5. Build a smaller pool of unmached fields.
6. Use fuzzy matching on the unresolved records only.
7. Create a method for manual review of low confidence matches.



## Core Python Libraries

- **pandas**: table wrangling and joins  
- **numpy**: missing values and simple numeric logic  
- **re**: text cleanup for IDs and addresses  
- **rapidfuzz**: fuzzy similarity scoring for unresolved pairs  


In [40]:
import re
import numpy as np
import pandas as pd
import os

try:
    from rapidfuzz import fuzz
except ImportError:
    raise ImportError("Install rapidfuzz first: pip install rapidfuzz")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


## Data configuration

By default this notebook uses a small toy example.

To adapt it to your real KC files:

1. set `USE_TOY_DATA = False`
2. point the file paths to your 2025 and 2026 exports
3. update the column names in the standardization cells if needed


In [41]:
USE_TOY_DATA = True

YEAR_2025_PATH = r"\\resfs.home.ku.edu\groups_hipaa\PSYC\kdsc_ClassData\KDSC-CDL-Project3\Data\2025_Building_Inventory_Master_File.xlsx"
YEAR_2026_PATH = r"\\resfs.home.ku.edu\groups_hipaa\PSYC\kdsc_ClassData\KDSC-CDL-Project3\Data\2026_Building_Inventory_Master_File.xlsx"

YEAR_2026_SKIPROWS = 5




The toy data below mirrors the kinds of differences that show up across years:

- slightly different building names,
- slightly different address formatting,
- missing IDs in some rows,
- and one clearly unresolved case.

This makes it easier to teach the join logic before applying it to the full files.


In [42]:
toy_2025 = pd.DataFrame({
    "Benchmarking ID": ["1001", "1002", np.nan, np.nan],
    "Building Name": ["Main Plaza", "River Center", "Oak Tower", "West Campus Annex"],
    "Building Address": ["100 Main Street", "200 River Rd", "3300 Oak Ave", "900 West Campus Dr"],
    "Building City": ["Kansas City", "Kansas City", "Kansas City", "Kansas City"],
    "Building State": ["MO", "MO", "MO", "MO"],
    "Building Zip": ["64105", "64106", "64111", "64114"]
})

toy_2026 = pd.DataFrame({
    "Benchmarking ID": ["1001", np.nan, np.nan, np.nan],
    "Property Name": ["Main Plaza Office", "River Center", "Oak TWR", "South Campus Annex"],
    "Address 1": ["100 MAIN ST.", "200 River Road", "3300 Oak Avenue", "910 West Campus Drive"],
    "City": ["Kansas City", "Kansas City", "Kansas City", "Kansas City"],
    "State/Province": ["MO", "MO", "MO", "MO"],
    "Postal Code": ["64105", "64106", "64111", "64114"]
})

display(toy_2025)
display(toy_2026)


,Benchmarking ID,Building Name,Building Address,Building City,Building State,Building Zip
0,1001,Main Plaza,100 Main Street,Kansas City,MO,64105
1,1002,River Center,200 River Rd,Kansas City,MO,64106
2,NaN,Oak Tower,3300 Oak Ave,Kansas City,MO,64111
3,NaN,West Campus Annex,900 West Campus Dr,Kansas City,MO,64114


,Benchmarking ID,Property Name,Address 1,City,State/Province,Postal Code
0,1001,Main Plaza Office,100 MAIN ST.,Kansas City,MO,64105
1,NaN,River Center,200 River Road,Kansas City,MO,64106
2,NaN,Oak TWR,3300 Oak Avenue,Kansas City,MO,64111
3,NaN,South Campus Annex,910 West Campus Drive,Kansas City,MO,64114


## Optional: load the full files instead

If you switch off the toy data, this cell shows one simple pattern for reading the two annual files.
 
The important part is that the two years often arrive with **different headers and slightly different formatting**, so we standardize them before comparing rows.


In [43]:
if USE_TOY_DATA:
    raw_2025 = toy_2025.copy()
    raw_2026 = toy_2026.copy()
else:
    raw_2025 = pd.read_excel(YEAR_2025_PATH)
    # The first 5 rows are empty. use skiprows to start at headers
    raw_2026 = pd.read_excel(YEAR_2026_PATH, skiprows=YEAR_2026_SKIPROWS)

print("2025 shape:", raw_2025.shape)
print("2026 shape:", raw_2026.shape)
display(raw_2025.head())
display(raw_2026.head())


2025 shape: (4, 6)
2026 shape: (4, 6)


,Benchmarking ID,Building Name,Building Address,Building City,Building State,Building Zip
0,1001,Main Plaza,100 Main Street,Kansas City,MO,64105
1,1002,River Center,200 River Rd,Kansas City,MO,64106
2,NaN,Oak Tower,3300 Oak Ave,Kansas City,MO,64111
3,NaN,West Campus Annex,900 West Campus Dr,Kansas City,MO,64114


,Benchmarking ID,Property Name,Address 1,City,State/Province,Postal Code
0,1001,Main Plaza Office,100 MAIN ST.,Kansas City,MO,64105
1,NaN,River Center,200 River Road,Kansas City,MO,64106
2,NaN,Oak TWR,3300 Oak Avenue,Kansas City,MO,64111
3,NaN,South Campus Annex,910 West Campus Drive,Kansas City,MO,64114


Looking at these dataframes, some headers have different naming convention (ie. 2025["Building Address"] and 2026["Address 1"]). To merge we will need to identify which headers we want from.
Benchmarking ID and address will be a good place to start, we can further use building name as a unique identifyer.

## Step 1: Standardize just enough columns to compare years

The full pipeline standardizes many fields.  
For teaching, we only keep the columns that matter most for linking:

- benchmarking ID
- building name
- address
- city/state/ZIP

We also add a `source_record_id` so each row stays traceable after concatenation.


In [44]:
# Setup dataframes with consistent column names and formats for matching
# # We make a new dataframe using dataframe.copy() and pull/rename the headers we want into a consistent format. 
# # # This way we can keep the original raw dataframes intact for reference.
df2025 = raw_2025.rename(columns={
    "Benchmarking ID": "benchmark_id",
    "Building Name": "building_name",
    "Building Address": "address",
    "Building City": "city",
    "Building State": "state",
    "Building Zip": "zip"
}).copy()

df2026 = raw_2026.rename(columns={
    "Benchmarking ID": "benchmark_id",
    "Property Name": "building_name",
    "Address 1": "address",
    "City": "city",
    "State/Province": "state",
    "Postal Code": "zip"
}).copy()

keep_cols = ["benchmark_id", "building_name", "address", "city", "state", "zip"]

df2025 = df2025[keep_cols].copy()
df2026 = df2026[keep_cols].copy()

# Add reporting year and source record ID for traceability
df2025["reporting_year"] = 2025
df2026["reporting_year"] = 2026

df2025["source_record_id"] = ["2025::" + str(i) for i in df2025.index]
df2026["source_record_id"] = ["2026::" + str(i) for i in df2026.index]




Now we have easuer to work with dataframes

In [45]:
display(df2025.head(3))
display(df2026.head(3))

,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id
0,1001,Main Plaza,100 Main Street,Kansas City,MO,64105,2025,2025::0
1,1002,River Center,200 River Rd,Kansas City,MO,64106,2025,2025::1
2,NaN,Oak Tower,3300 Oak Ave,Kansas City,MO,64111,2025,2025::2


,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id
0,1001,Main Plaza Office,100 MAIN ST.,Kansas City,MO,64105,2026,2026::0
1,NaN,River Center,200 River Road,Kansas City,MO,64106,2026,2026::1
2,NaN,Oak TWR,3300 Oak Avenue,Kansas City,MO,64111,2026,2026::2


From these subset dataframes we can see how complete they are

In [46]:
# Columns we want to check for completeness
norm_cols = [
    "benchmark_id",
    "building_name",
    "address",
    "city",
    "state",
    "zip"
]

# Function to summarize completeness for one dataframe
def completeness_summary(df, label):
    summary = pd.DataFrame({
        "non_missing_count": df[norm_cols].notna().sum(),
        "missing_count": df[norm_cols].isna().sum(),
        "percent_complete": (df[norm_cols].notna().mean() * 100).round(2)
    })
    summary["dataset"] = label
    return summary.reset_index().rename(columns={"index": "column"})

# Build summaries for both years
comp2025 = completeness_summary(df2025, "2025")
comp2026 = completeness_summary(df2026, "2026")

# Combine into one table
completeness_results = pd.concat([comp2025, comp2026], ignore_index=True)

display(completeness_results)

,column,non_missing_count,missing_count,percent_complete,dataset
0,benchmark_id,2,2,50.0,2025
1,building_name,4,0,100.0,2025
2,address,4,0,100.0,2025
3,city,4,0,100.0,2025
4,state,4,0,100.0,2025
5,zip,4,0,100.0,2025
6,benchmark_id,1,3,25.0,2026
7,building_name,4,0,100.0,2026
8,address,4,0,100.0,2026
9,city,4,0,100.0,2026


## Step 2: Normalize the values used for matching

This is one of the most important ideas in the workflow. The data are coming from a messy state, and likely have errors in spelling or formatting. 

We do **not** want addresses like `"100 Main Street"` and `"100 MAIN ST."` to behave like different buildings.  
So before we join, we create cleaned versions of the fields we expect to compare.


### Short explanation of pandas `.str` functions

We can use pandas string functions with `.str` to normalize data before matching or comparing values.  
This helps make the formatting consistent across columns like names, addresses, cities, states, and ZIP codes.

**`.str.upper()`**  
Converts all text to uppercase so values match more consistently.  
Example: `"Main Street"` → `"MAIN STREET"`

**`.str.replace()`**  
Replaces part of a string with something else.  
This is useful for removing punctuation, shortening words, or cleaning formatting.

Examples:
- `str.replace(r"[^A-Z0-9]", "", regex=True)` removes anything that is not a capital letter or number
- `str.replace(r"\bSTREET\b", "ST", regex=True)` changes `"STREET"` to `"ST"`
- `str.replace(r"\s+", " ", regex=True)` replaces multiple spaces with one space

**`.str.strip()`**  
Removes spaces at the beginning and end of a string.  
Example: `"  BOSTON  "` → `"BOSTON"`

**`.str[:5]`**  
Keeps only the first 5 characters.  
Used here to keep ZIP codes in 5-digit form.

**`.str.cat()`**  
Combines strings together.  
Used here to create matching keys from multiple cleaned columns.

---


In [47]:
for df in [df2025, df2026]:
    # For each of the fields, we create a new normalized version of the field with consistent formatting and cleaned values to improve matching accuracy.
    df["benchmark_id_clean"] = (
        df["benchmark_id"]
        .astype("string")
        .str.upper()
        .str.replace(r"[^A-Z0-9]", "", regex=True)
        .str.strip()
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["name_norm"] = (
        df["building_name"]
        .astype("string")
        .str.upper()
        .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["address_norm"] = (
        df["address"]
        .astype("string")
        .str.upper()
        .str.replace(r"\bSTREET\b", "ST", regex=True)
        .str.replace(r"\bAVENUE\b", "AVE", regex=True)
        .str.replace(r"\bROAD\b", "RD", regex=True)
        .str.replace(r"\bDRIVE\b", "DR", regex=True)
        .str.replace(r"[^A-Z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["city_norm"] = (
        df["city"]
        .astype("string")
        .str.upper()
        .str.strip()
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["state_norm"] = (
        df["state"]
        .astype("string")
        .str.upper()
        .str.strip()
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["zip_norm"] = (
        df["zip"]
        .astype("string")
        .str.replace(r"[^0-9]", "", regex=True)
        .str[:5]
        .mask(lambda s: s.eq(""), pd.NA)
    )

    df["addr_zip_key"] = df["address_norm"].str.cat(df["zip_norm"], sep="|")
    df["name_addr_zip_key"] = df["name_norm"].str.cat(df["addr_zip_key"], sep="|")

## Step 3: Create conservative exact-match keys

### Why multiple keys?

No single field is complete enough for every row.

- `benchmark_id_clean` works well when the ID is present.
- `addr_zip_key` helps when IDs are missing but the address is stable.
- `name_addr_zip_key` is a slightly stricter version for cases with repeated addresses.

We try exact keys first because they are easier to defend than fuzzy matches.


In [48]:
for df in [df2025, df2026]:
    # Create a combined key using the normalized address and normalized ZIP code.
    # str.cat() joins the two string columns together, and sep="|" adds a separator
    df["addr_zip_key"] = df["address_norm"].str.cat(df["zip_norm"], sep="|")
    df["name_addr_zip_key"] = df["name_norm"].str.cat(df["addr_zip_key"], sep="|")

display(
    df2025[["source_record_id", "benchmark_id_clean","addr_zip_key", "name_addr_zip_key"]]
)
display(
    df2026[["source_record_id", "benchmark_id_clean","addr_zip_key", "name_addr_zip_key"]]
)


,source_record_id,benchmark_id_clean,addr_zip_key,name_addr_zip_key
0,2025::0,1001,100 MAIN ST|64105,MAIN PLAZA|100 MAIN ST|64105
1,2025::1,1002,200 RIVER RD|64106,RIVER CENTER|200 RIVER RD|64106
2,2025::2,<NA>,3300 OAK AVE|64111,OAK TOWER|3300 OAK AVE|64111
3,2025::3,<NA>,900 WEST CAMPUS DR|64114,WEST CAMPUS ANNEX|900 WEST CAMPUS DR|64114


,source_record_id,benchmark_id_clean,addr_zip_key,name_addr_zip_key
0,2026::0,1001,100 MAIN ST|64105,MAIN PLAZA OFFICE|100 MAIN ST|64105
1,2026::1,<NA>,200 RIVER RD|64106,RIVER CENTER|200 RIVER RD|64106
2,2026::2,<NA>,3300 OAK AVE|64111,OAK TWR|3300 OAK AVE|64111
3,2026::3,<NA>,910 WEST CAMPUS DR|64114,SOUTH CAMPUS ANNEX|910 WEST CAMPUS DR|64114


## Step 4: Exact join by benchmarking ID

This is usually the cleanest link when the ID is present in both years.

----Can put a little bit here based on the missingness of the columns---


In [49]:
exact_by_benchmark = (
    df2025[df2025["benchmark_id_clean"].notna()]
    .merge(
        df2026[df2026["benchmark_id_clean"].notna()],
        on="benchmark_id_clean",
        how="inner",
        suffixes=("_2025", "_2026")
    )
)

display(exact_by_benchmark[[
    "benchmark_id_clean",
    "source_record_id_2025", "building_name_2025", "address_2025",
    "source_record_id_2026", "building_name_2026", "address_2026"
]])


,benchmark_id_clean,source_record_id_2025,building_name_2025,address_2025,source_record_id_2026,building_name_2026,address_2026
0,1001,2025::0,Main Plaza,100 Main Street,2026::0,Main Plaza Office,100 MAIN ST.


## Step 5: Exact join by normalized address + ZIP

In this step, we first keep track of the records that were already matched using `benchmark_id`.

Then, we remove those matched records from both datasets so we only work with the remaining unmatched rows.

Next, we match the remaining records using `addr_zip_key`, to try and further the matching.



In [50]:
matched_2025 = set()
matched_2026 = set()

# Add the record IDs that were already matched exactly by benchmark ID
matched_2025 = matched_2025.union(set(exact_by_benchmark["source_record_id_2025"]))
matched_2026 = matched_2026.union(set(exact_by_benchmark["source_record_id_2026"]))

# Keep only records that have not been matched yet
unmatched_2025 = df2025[~df2025["source_record_id"].isin(matched_2025)].copy()
unmatched_2026 = df2026[~df2026["source_record_id"].isin(matched_2026)].copy()

# Match the remaining records using the normalized address + ZIP key
exact_by_address = (
    unmatched_2025[unmatched_2025["addr_zip_key"].notna()]
    .merge(
        unmatched_2026[unmatched_2026["addr_zip_key"].notna()],
        on="addr_zip_key",
        how="inner",
        suffixes=("_2025", "_2026")
    )
)

# Display the address-based matches so we can review them
display(exact_by_address[[
    "addr_zip_key",
    "source_record_id_2025", "building_name_2025", "address_2025",
    "source_record_id_2026", "building_name_2026", "address_2026"
]])

,addr_zip_key,source_record_id_2025,building_name_2025,address_2025,source_record_id_2026,building_name_2026,address_2026
0,200 RIVER RD|64106,2025::1,River Center,200 River Rd,2026::1,River Center,200 River Road
1,3300 OAK AVE|64111,2025::2,Oak Tower,3300 Oak Ave,2026::2,Oak TWR,3300 Oak Avenue


## Why fuzzy matching is not the first step

After the exact joins above, we are left with a much smaller unresolved pool.

Afterwards we can use fuzzy matching more effectively:

- fewer rows to compare,
- fewer false positives,
- and easier review.

In other words, fuzzy matching is used as a **fallback**, not as the default merge strategy.


In [51]:
# Add the record IDs that were matched in the address-based step
matched_2025 = matched_2025.union(set(exact_by_address["source_record_id_2025"]))
matched_2026 = matched_2026.union(set(exact_by_address["source_record_id_2026"]))

# Rebuild the unmatched datasets by removing everything matched so far
unmatched_2025 = df2025[~df2025["source_record_id"].isin(matched_2025)].copy()
unmatched_2026 = df2026[~df2026["source_record_id"].isin(matched_2026)].copy()

# Show how many records are still left unmatched in each dataset
print("Still unmatched in 2025:", len(unmatched_2025))
print("Still unmatched in 2026:", len(unmatched_2026))

# Display the remaining unmatched records for review
display(unmatched_2025)
display(unmatched_2026)

Still unmatched in 2025: 1
Still unmatched in 2026: 1


,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id,benchmark_id_clean,name_norm,address_norm,city_norm,state_norm,zip_norm,addr_zip_key,name_addr_zip_key
3,NaN,West Campus Annex,900 West Campus Dr,Kansas City,MO,64114,2025,2025::3,<NA>,WEST CAMPUS ANNEX,900 WEST CAMPUS DR,KANSAS CITY,MO,64114,900 WEST CAMPUS DR|64114,WEST CAMPUS ANNEX|900 WEST CAMPUS DR|64114


,benchmark_id,building_name,address,city,state,zip,reporting_year,source_record_id,benchmark_id_clean,name_norm,address_norm,city_norm,state_norm,zip_norm,addr_zip_key,name_addr_zip_key
3,NaN,South Campus Annex,910 West Campus Drive,Kansas City,MO,64114,2026,2026::3,<NA>,SOUTH CAMPUS ANNEX,910 WEST CAMPUS DR,KANSAS CITY,MO,64114,910 WEST CAMPUS DR|64114,SOUTH CAMPUS ANNEX|910 WEST CAMPUS DR|64114


### Save tutorial 1 outputs


In [52]:
from pathlib import Path
output_dir = Path.cwd() / "tutorial_output_csv"
output_dir.mkdir(parents=True, exist_ok=True)

df2025.to_csv(output_dir / "df2025.csv", index=False)
df2026.to_csv(output_dir / "df2026.csv", index=False)
exact_by_benchmark.to_csv(output_dir / "exact_by_benchmark.csv", index=False)
exact_by_address.to_csv(output_dir / "exact_by_address.csv", index=False)
unmatched_2025.to_csv(output_dir / "unmatched_2025.csv", index=False)
unmatched_2026.to_csv(output_dir / "unmatched_2026.csv", index=False)

print(f"Saved tutorial 1 CSV outputs to: {output_dir.resolve()}")
print(sorted([p.name for p in output_dir.glob('*.csv')]))

Saved tutorial 1 CSV outputs to: C:\Users\Joey\Desktop\github\KC_Cotality\curriculum\tutorial_output_csv
['df2025.csv', 'df2026.csv', 'exact_by_address.csv', 'exact_by_benchmark.csv', 'unmatched_2025.csv', 'unmatched_2026.csv']


## Notebook summary

In this notebook, we prepared the 2025 and 2026 datasets for merging by:

- checking key columns and their completeness
- cleaning and normalizing important fields such as benchmark ID, building name, address, city, state, and ZIP code
- creating combined matching keys from normalized values
- finding exact matches first by `benchmark_id`
- matching additional records by normalized address and ZIP code
- keeping track of matched and unmatched records after each step

In Tutorial 2, we will continue with fuzzy matching and manual review so we can create a more complete unified dataset.

## Tutorial Questions

- Why do we normalize text before trying to match records?
- How can differences in capitalization, punctuation, or spacing cause matching problems?
- Why might an address-based match work when a building name match does not?
- Why is it useful to match records in stages instead of trying every method at once?
- Why might we trust an exact benchmark ID match more than a fuzzy name match?
- Why is manual review still important even after automated matching steps?